# MCP Server Knowledge Source Quickstart

This notebook demonstrates the MCP Server Knowledge Source path using the public Microsoft Learn MCP endpoint:

```text
https://learn.microsoft.com/api/mcp
```

Deployment mode mapping: this notebook is the guided tutorial for `mcp-only`, and it is also the first validation step before `byo-fabric` or `full`.

It proves the minimum live loop:

1. Build an MCP Server Knowledge Source payload.
2. Build an MCP-only Knowledge Base payload.
3. Optionally create/update both resources in Azure AI Search.
4. Optionally retrieve from the Knowledge Base.
5. Inspect `activity`, `references`, and `sourceData`.

Primary manual: https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server

## Runtime Safety

The notebook defaults to dry-run mode. It prints payloads and inspects checked-in offline responses.

To make live calls, set:

```text
RUN_LIVE_CALLS=true
SEARCH_ENDPOINT=https://<search-service>.search.windows.net
SEARCH_API_KEY=<search-admin-or-query-key>
AZURE_OPENAI_ENDPOINT=https://<azure-openai-or-foundry-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT_ID=<chat-completions-deployment-name>
AZURE_OPENAI_MODEL_NAME=<model-name>
```

For private local testing, set `LIVE_KS_ENV_FILE=/path/to/.env`. Do not commit live env files.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from ks_factory import create_knowledge_base, create_mcp_server_knowledge_source
from ks_factory.diagnostics import summarize_activity, summarize_references


def load_env_file(path: str | None) -> None:
    if not path:
        return
    env_path = Path(path).expanduser()
    if not env_path.exists():
        raise FileNotFoundError(env_path)
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(os.getenv("LIVE_KS_ENV_FILE"))

SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT", "https://<search-service>.search.windows.net").rstrip("/")
SEARCH_API_VERSION = os.getenv("SEARCH_API_VERSION", "2026-05-01-preview")
SEARCH_API_KEY = os.getenv("SEARCH_API_KEY", "<search-admin-or-query-key-for-poc>")
RUN_LIVE_CALLS = os.getenv("RUN_LIVE_CALLS", "false").lower() == "true"

MCP_KS_NAME = os.getenv("MCP_KNOWLEDGE_SOURCE_NAME", "microsoft-learn-mcp-ks")
MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", "https://learn.microsoft.com/api/mcp")
MCP_TOOL_NAME = os.getenv("MCP_TOOL_NAME", "microsoft_docs_search")
KNOWLEDGE_BASE_NAME = os.getenv("KNOWLEDGE_BASE_NAME", "live-knowledge-sources-kb")

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "https://<azure-openai-or-foundry-resource>.openai.azure.com")
AZURE_OPENAI_DEPLOYMENT_ID = os.getenv("AZURE_OPENAI_DEPLOYMENT_ID", "<chat-completions-deployment-name>")
AZURE_OPENAI_MODEL_NAME = os.getenv("AZURE_OPENAI_MODEL_NAME", "<model-name>")

print("RUN_LIVE_CALLS =", RUN_LIVE_CALLS)
print("MCP server =", MCP_SERVER_URL)
print("MCP tool =", MCP_TOOL_NAME)

## Build The MCP Payloads

These payloads match the REST flow in:

```text
samples/rest/01-create-mcp-server-ks.http
samples/rest/02-create-mcp-only-kb.http
samples/rest/03-retrieve-mcp.http
```

In [ ]:
mcp_knowledge_source = create_mcp_server_knowledge_source(
    name=MCP_KS_NAME,
    server_url=MCP_SERVER_URL,
    tool_name=MCP_TOOL_NAME,
    description="Microsoft Learn MCP grounding source for official documentation.",
)

mcp_only_knowledge_base = create_knowledge_base(
    name=KNOWLEDGE_BASE_NAME,
    knowledge_source_names=[MCP_KS_NAME],
    description="Knowledge Base for validating MCP Server live grounding before adding tenant-specific sources.",
    retrieval_instructions=(
        "Use the Microsoft Learn MCP Server knowledge source for questions about "
        "Azure AI Search, Foundry IQ, and Microsoft documentation."
    ),
    azure_openai_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_openai_deployment_id=AZURE_OPENAI_DEPLOYMENT_ID,
    azure_openai_model_name=AZURE_OPENAI_MODEL_NAME,
)

print(json.dumps({"knowledgeSource": mcp_knowledge_source, "knowledgeBase": mcp_only_knowledge_base}, indent=2))

## Optional Live Calls

This cell only executes when `RUN_LIVE_CALLS=true`. The retrieve query is intentionally documentation-oriented so the Microsoft Learn MCP tool has a clear job.

In [ ]:
def search_url(path: str) -> str:
    if "<" in SEARCH_ENDPOINT:
        raise ValueError("Set SEARCH_ENDPOINT before running live calls.")
    return f"{SEARCH_ENDPOINT}{path}?api-version={SEARCH_API_VERSION}"


def request_json(method: str, path: str, body: dict[str, Any] | None = None) -> dict[str, Any]:
    headers = {
        "api-key": SEARCH_API_KEY,
        "Content-Type": "application/json",
        "Prefer": "return=representation",
    }
    data = None if body is None else json.dumps(body).encode("utf-8")
    request = urllib.request.Request(search_url(path), data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{method} {path} failed: {error.code}\n{detail}") from error


retrieve_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What must be configured to create an Azure AI Search MCP Server knowledge source?",
                }
            ],
        }
    ],
    "includeActivity": True,
    "knowledgeSourceParams": [
        {
            "kind": "mcpServer",
            "knowledgeSourceName": MCP_KS_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
        }
    ],
    "outputMode": "answerSynthesis",
    "retrievalReasoningEffort": {"kind": "low"},
    "maxRuntimeInSeconds": 60,
}

if RUN_LIVE_CALLS:
    request_json("PUT", f"/knowledgesources/{MCP_KS_NAME}", mcp_knowledge_source)
    request_json("PUT", f"/knowledgebases/{KNOWLEDGE_BASE_NAME}", mcp_only_knowledge_base)
    live_response = request_json("POST", f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/retrieve", retrieve_body)
    print("Activity")
    print(json.dumps(summarize_activity(live_response), indent=2))
    print("References")
    print(json.dumps(summarize_references(live_response), indent=2))
else:
    print("Dry run. Set RUN_LIVE_CALLS=true to create resources and retrieve live results.")
    print(json.dumps(retrieve_body, indent=2))

## Offline Replay

Use the checked-in sample response to validate the expected MCP trace shape without keys or network calls.

In [ ]:
offline_response_path = REPO_ROOT / "samples" / "responses" / "mcp-retrieve.sample.json"
offline_response = json.loads(offline_response_path.read_text(encoding="utf-8"))

print("Activity")
print(json.dumps(summarize_activity(offline_response), indent=2))
print("References")
print(json.dumps(summarize_references(offline_response), indent=2))

## Expected Result

A successful MCP run should show:

- `activity[*].type == "mcpServer"`
- `knowledgeSourceName == "microsoft-learn-mcp-ks"`
- `toolName == "microsoft_docs_search"`
- MCP references with `sourceData`

After this path works, validate Fabric Ontology KS in `02-fabric-ontology-ks-airline-ops.ipynb`.